In [ ]:
# Standard library imports
import sys
import urllib.request
from pathlib import Path

# Third party imports
import joblib
import pandas as pd

### Run configuration

In [ ]:
# Flag to control environment-specific paths & configurations
KAGGLE = False

# Checkpoint selector: 'latest' or specific number like '003', '006', etc.
CHECKPOINT = 'latest'

### Add ensemble_classifier module to path

In [ ]:
# Add path to ensemble_classifier module
import json

if KAGGLE:
    # On Kaggle, the module should be uploaded as part of the checkpoint dataset
    # Dataset name pattern: diabetes-ensemble-checkpoint-XXX
    checkpoint_dataset = f'diabetes-ensemble-checkpoint-{CHECKPOINT}'
    module_path = Path(f'/kaggle/input/{checkpoint_dataset}')
else:
    # For local/GitHub, find the checkpoint directory
    models_dir = Path('../models')
    run_dirs = sorted([d for d in models_dir.glob('run_*') if d.is_dir()], reverse=True)
    
    if len(run_dirs) == 0:
        raise FileNotFoundError("No ensemble model runs found. Train a model first.")
    
    # Use most recent run
    latest_run = run_dirs[0]
    checkpoints_dir = latest_run / 'checkpoints'
    
    if not checkpoints_dir.exists():
        raise FileNotFoundError(
            f"No checkpoints directory found at {checkpoints_dir}. "
            "Run training to create checkpoints."
        )
    
    # Find checkpoint
    if CHECKPOINT == 'latest':
        # Find highest numbered checkpoint
        checkpoint_dirs = sorted(checkpoints_dir.glob('checkpoint_*'))
        if not checkpoint_dirs:
            raise FileNotFoundError(f"No checkpoints found in {checkpoints_dir}")
        module_path = checkpoint_dirs[-1]
    else:
        # Use specified checkpoint number
        module_path = checkpoints_dir / f'checkpoint_{CHECKPOINT}'
        if not module_path.exists():
            raise FileNotFoundError(f"Checkpoint not found: {module_path}")

sys.path.insert(0, str(module_path))

# Import ensemble classifier (needed for model deserialization)
from ensemble_classifier import EnsembleClassifier

print(f"Checkpoint: {CHECKPOINT}")
print(f"Module path: {module_path}")
print(f"EnsembleClassifier imported successfully")

## 1. Asset loading

In [ ]:
# Set file paths based on environment
if KAGGLE:
    # Kaggle paths - data is in /kaggle/input/
    test_df_path = '/kaggle/input/playground-series-s5e12/test.csv'
    checkpoint_dataset = f'diabetes-ensemble-checkpoint-{CHECKPOINT}'
    model_path = Path(f'/kaggle/input/{checkpoint_dataset}/ensemble_stage1_models.joblib')
    stage2_model_path = Path(f'/kaggle/input/{checkpoint_dataset}/stage2_model.h5')
    metadata_path = Path(f'/kaggle/input/{checkpoint_dataset}/metadata.json')
else:
    # Local paths
    test_df_path = 'https://gperdrizet.github.io/FSA_devops/assets/data/unit3/diabetes_prediction_test.csv'
    
    # Checkpoint paths already determined in previous cell (module_path)
    model_path = module_path / 'ensemble_stage1_models.joblib'
    stage2_model_path = module_path / 'stage2_model.h5'
    metadata_path = module_path / 'metadata.json'
    
    # Verify checkpoint files exist
    for path in [model_path, stage2_model_path, metadata_path]:
        if not path.exists():
            raise FileNotFoundError(f"Required file not found: {path}")

print(f"Loading test data from: {test_df_path}")
print(f"Loading model from: {model_path}")
print(f"Stage 2 model path: {stage2_model_path}")

# Load the testing dataset
test_df = pd.read_csv(test_df_path)
print(f"\nTest data shape: {test_df.shape}")

# Load checkpoint metadata
with open(metadata_path) as f:
    metadata = json.load(f)

print(f"\n{'='*60}")
print(f"Checkpoint Metadata:")
print(f"  Checkpoint: {metadata['checkpoint_num']:03d}")
print(f"  Timestamp: {metadata['timestamp']}")
print(f"  Ensemble size: {metadata['ensemble_size']} models")
print(f"  Stage 1 AUC: {metadata['stage1_val_auc']:.4f}")
print(f"  Stage 2 AUC: {metadata['stage2_val_auc']:.4f}")
print(f"  Retraining count: {metadata['retraining_count']}")
if metadata.get('pseudo_labeling', {}).get('enabled'):
    print(f"  Pseudo-labeling: enabled")
print(f"{'='*60}\n")

# Load the ensemble model
model = joblib.load(model_path)
print(f"Model loaded: {model}")

# Set Stage 2 model path (triggers lazy loading on first predict)
model.stage2_model_path = str(stage2_model_path)
print(f"Stage 2 model path configured")

# Display first few rows
test_df.head()

## 2. Inference

In [ ]:
print("Running inference...")
print(f"  Processing {len(test_df):,} samples through {model.n_models_} ensemble models")

# Make probability predictions (not class labels)
predictions_proba = model.predict_proba(test_df)

# Extract probability of positive class (diabetes = 1)
predictions = predictions_proba[:, 1]

# Create submission dataframe with probabilities
predictions_df = pd.DataFrame({
    'id': test_df['id'].astype(int),
    'diagnosed_diabetes': predictions  # Float probabilities, not int labels
})

print(f"\nPredictions complete!")
print(f"  Prediction statistics:")
print(f"    Min:  {predictions.min():.4f}")
print(f"    Max:  {predictions.max():.4f}")
print(f"    Mean: {predictions.mean():.4f}")
print(f"    Std:  {predictions.std():.4f}")

predictions_df.head(10)

## 3. Save submission file

In [ ]:
# Set submission file path based on environment
if KAGGLE:
    submission_path = Path('submission.csv')
else:
    # Create data directory if it doesn't exist
    data_dir = Path('../data')
    data_dir.mkdir(parents=True, exist_ok=True)
    submission_path = data_dir / 'ensemble_submission.csv'

# Save submission file
predictions_df.to_csv(submission_path, index=False)
print(f'Submission saved to: {submission_path}')
print(f'File size: {submission_path.stat().st_size / 1024:.1f} KB')

## 4. Summary

In [ ]:
print("=" * 80)
print("INFERENCE COMPLETE")
print("=" * 80)
print(f"Model: {model}")
print(f"Samples processed: {len(predictions_df):,}")
print(f"Submission file: {submission_path}")
print("=" * 80)